In [0]:
from pyspark.sql.functions import col, upper, when, round, count, lit, current_timestamp, isnan
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Leer todas las versiones (ambos lotes concatenados)
df_cg = spark.read.format("delta").load("/Volumes/cryptolake/bronze/data/coingecko")
df_cc = spark.read.format("delta").load("/Volumes/cryptolake/bronze/data/coincap")

print(f"Registros CoinGecko: {df_cg.count()}")
print(f"Registros CoinCap: {df_cc.count()}")

# Función para reportar nulos
def check_nulls(df, name):
    print(f"\n--- Nulos en {name} ---")
    for c in df.columns:
        null_count = df.filter(col(c).isNull()).count()
        print(f"{c}: {null_count}")

check_nulls(df_cg, "CoinGecko")
check_nulls(df_cc, "CoinCap")

# Eliminar filas con nulos en columnas críticas para el JOIN
df_cg_clean = df_cg.dropna(subset=["symbol", "price_usd", "market_cap", "volume_24h"])
df_cc_clean = df_cc.dropna(subset=["symbol", "price_usd", "market_cap_usd", "volume_24h"])

# Estandarizar símbolos a mayúsculas
df_cg_clean = df_cg_clean.withColumn("symbol_upper", upper(col("symbol")))
df_cc_clean = df_cc_clean.withColumn("symbol_upper", upper(col("symbol")))

In [0]:
# LEFT JOIN para no perder monedas solo en CoinGecko
df_joined = df_cg_clean.join(
    df_cc_clean.select(
        "symbol_upper",
        col("volume_24h").alias("cc_volume"),
        "market_cap_usd"
    ),
    on="symbol_upper",
    how="left"
)

# liquidity_ratio = volumen_24h_CoinCap / market_cap_usd
df_silver = df_joined.withColumn(
    "liquidity_ratio",
    when(col("cc_volume").isNotNull(), round(col("cc_volume") / col("market_cap_usd"), 6))
    .otherwise(None)
)

# Seleccionar y renombrar columnas finales
final_cols = [
    "symbol_upper", "name", "price_usd", "market_cap", "volume_24h",
    "percent_change_24h", "ingestion_ts", "source_file", "liquidity_ratio"
]
df_silver = df_silver.select(final_cols).withColumnRenamed("symbol_upper", "symbol")

df_silver.show(5, truncate=False)

In [0]:
silver_path = "/Volumes/cryptolake/silver/data/crypto_assets"

df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(silver_path)

print(f"✅ Tabla Silver guardada en {silver_path}")

In [0]:
from pyspark.sql.functions import col

# Rutas correctas de tus volúmenes
bronze_path = "/Volumes/cryptolake/bronze/data/coingecko"

# Ver historial de versiones
print("Historial de la tabla Bronze CoinGecko:")
display(spark.sql(f"DESCRIBE HISTORY delta.`{bronze_path}`"))

# Leer versiones 0 y 1 (asumiendo que tienes al menos dos lotes)
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(bronze_path)
df_v1 = spark.read.format("delta").option("versionAsOf", 1).load(bronze_path)

print(f"Registros en versión 0 (primer lote): {df_v0.count()}")
print(f"Registros en versión 1 (segundo lote): {df_v1.count()}")

# Comparar precio de Bitcoin entre versiones
btc_v0 = df_v0.filter(col("symbol") == "btc").select("price_usd", "ingestion_ts").collect()
btc_v1 = df_v1.filter(col("symbol") == "btc").select("price_usd", "ingestion_ts").collect()

print("BTC precio lote 1:", btc_v0)
print("BTC precio lote 2:", btc_v1)

In [0]:
from pyspark.sql.functions import lit   # Asegurar que lit está importado

# Ruta correcta de tu volumen Bronze
bronze_path = "/Volumes/cryptolake/bronze/data/coingecko"

# Simular un nuevo lote con columna extra "ath" (all-time high)
df_new = df_cg_clean.limit(10).withColumn("ath", lit(69000.0))

# Escribir en la misma ubicación Bronze con mergeSchema = true
df_new.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(bronze_path)

# Verificar que la tabla ahora tiene la columna "ath"
df_updated = spark.read.format("delta").load(bronze_path)
print("Columnas después de schema evolution:", df_updated.columns)

## Evidencias del Módulo 3 – Pipeline Batch (Bronze → Silver)

### Requisitos cumplidos

1. **Notebook Bronze**: Ingesta de dos lotes de CoinGecko y dos lotes de CoinCap. Cada registro incluye `ingestion_ts` (timestamp automático de carga) y `source_file` (identificador del lote).

2. **Notebook Silver**:  
   - Validación de nulos en columnas clave (symbol, price_usd, market_cap, volume_24h).  
   - JOIN entre CoinGecko y CoinCap usando `symbol` en mayúsculas.  
   - Cálculo de `liquidity_ratio = volume_24h_CoinCap / market_cap_usd`.  
   - Escritura en Delta Lake con la opción `.option("mergeSchema", "true")`.

3. **Schema Evolution**: Se agregó la columna `ath` a un nuevo lote en la tabla Bronze y se escribió con `mergeSchema = true`. La tabla Bronze ahora incluye la columna `ath` sin errores.

4. **Time Travel**: Se consultaron las versiones 0 y 1 de la tabla Bronze de CoinGecko, demostrando diferentes precios de BTC y diferentes timestamps de ingestión.

5. **Rutas de almacenamiento (Unity Catalog Volumes)**:  
   - Bronze: `/Volumes/cryptolake/bronze/data/coingecko` y `/Volumes/cryptolake/bronze/data/coincap`  
   - Silver: `/Volumes/cryptolake/silver/data/crypto_assets`

### Comandos útiles para la verificación

```sql
-- Ver historial de versiones de la tabla Bronze (CoinGecko)
DESCRIBE HISTORY delta.`/Volumes/cryptolake/bronze/data/coingecko`;

-- Time Travel: consultar versión 0 (primer lote)
SELECT * FROM delta.`/Volumes/cryptolake/bronze/data/coingecko` VERSION AS OF 0;

-- Ver datos de la capa Silver
SELECT * FROM delta.`/Volumes/cryptolake/silver/data/crypto_assets` LIMIT 10;

### Nota sobre el almacenamiento

El entorno de Databricks Free Edition no permite escritura en la raíz de DBFS. Por esta razón, los datos se almacenan en volúmenes de Unity Catalog, que cumplen la misma función de persistencia y permiten todas las operaciones requeridas (Delta Lake, Time Travel, Schema Evolution). Las rutas equivalentes son:

- Bronze: `/Volumes/cryptolake/bronze/data/coingecko` y `/Volumes/cryptolake/bronze/data/coincap`
- Silver: `/Volumes/cryptolake/silver/data/crypto_assets`

Esto no afecta la funcionalidad ni los resultados del pipeline.